This file merges data from different files in preparation for analyses

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

d = f"{os.getcwd()}/"
PATH_RAW = os.path.join(d, "..", "raw")   # raw inputs (downloads / intermediate CSVs)
PATH_DATA = os.path.join(d, "..", "data")    # processed modeling tables
# Output folder for model results (forecasts, performance tables)
PATH_OUTPUT = os.path.join(d, "..", "output")
# Create folder if it does not exist
os.makedirs(PATH_OUTPUT, exist_ok=True)
data_file = "data.csv"
meta_file = "meta.csv"

df = pd.read_csv(f"{PATH_DATA}/{data_file}" , index_col='date', parse_dates=True)

df.head(5)

The dataset includes the output variable, GDP

In [ ]:
df['RGDP0000'].dropna().plot()

We wish to include **google trends** and **nighttime lights** data on our main data

First, let's keep only GDP.

In [ ]:
df = df[['RGDP0000']].dropna()
df.head(5)

## Google Trends

1. Load data
2. Apply seasonal adjustment
3. Resample

In [ ]:
df_gt = pd.read_csv(f"{PATH_RAW}/gtrends.csv", parse_dates=True, index_col='date')
df_gt.head(5)

Keep only data with enough observations

In [ ]:
# Transform zeroes into np.nan
df_gt = df_gt.replace(0, np.nan)

# drop columns with too many (75%) nan
df_gt = df_gt.dropna(axis=1, thresh=0.25 * len(df_gt))

# Generate a column with the average
df_gt['gt_mean'] = df_gt.mean(axis=1)

In [ ]:
df_gt['gt_mean']

In [ ]:
df_gt['gt_mean'].plot()

Keep only the mean variable

In [ ]:
df_gt = df_gt[['gt_mean']]
df_gt.head(5)

**Seasonal Adjustment**

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

seasonal_decompose(
    df_gt["gt_mean"], model='additive', extrapolate_trend='freq', period=12
    ).plot();

`seasonal_decompose()` splits a time series into its structural parts. The key arguments here are:

- **`model='additive'`**: assumes the components add together — appropriate when the seasonal swings stay roughly the same size over time. Use `'multiplicative'` if they grow with the level of the series.
- **`period=12`**: tells the function one seasonal cycle = 12 observations (i.e. one year of monthly data).
- **`extrapolate_trend='freq'`**: the trend is estimated with a moving average, which leaves `NaN` at the first and last few observations. This fills those gaps by linearly extrapolating from the nearest valid values.

The decomposition assumes:

$$Y_t = \text{Trend}_t + \text{Seasonal}_t + \text{Residual}_t$$

- **Trend**: the long-run direction of the series
- **Seasonal**: a repeating pattern within each year
- **Residual**: what is left after removing trend and seasonality

We seasonally adjust by subtracting the seasonal component:

$$\text{Adjusted}_t = Y_t - \text{Seasonal}_t = \text{Trend}_t + \text{Residual}_t$$

In [ ]:
df_gt["gt_mean_adjusted"] = df_gt["gt_mean"] - seasonal_decompose(                                                          
      df_gt["gt_mean"].dropna(), model='additive', extrapolate_trend='freq', period=12                 
  ).seasonal

df_gt.head(5)

Plot google trends data and compare the unadjusted and adjusted series

Keep column of interest and rename

In [ ]:
# Keep only adjusted data
df_gt = df_gt[[ col for col in df_gt.columns if "adjusted" in col or "RGDP" in col ]]
# Eliminate _adjusted suffix
df_gt.columns = [ col.replace("_adjusted","") for col in df_gt.columns ]
df_gt.head(5)

## NTL data

Load data_ntl_shape.csv and repeat

In [ ]:
df_ntl = pd.read_csv(f"{PATH_RAW}/data_ntl_shape.csv", parse_dates=True, index_col='date')
df_ntl = df_ntl[['ntl_mean']]
# We first resample to monthly frequency, then apply seasonal adjustment
df_ntl = df_ntl.resample('MS').mean()
df_ntl.head(5)

**Seasonal adjustment:** Apply the same code as above

In [ ]:
seasonal_decompose().plot();

In [ ]:
df_ntl["ntl_mean_adjusted"] = 

df_ntl.head(5)

In [ ]:
df_ntl.plot()

In [ ]:
# Keep only adjusted data
df_ntl = df_ntl[[ col for col in df_ntl.columns if "adjusted" in col or "RGDP" in col ]]
# Eliminate _adjusted suffix
df_ntl.columns = [ col.replace("_adjusted","") for col in df_ntl.columns ]
df_ntl.head(5)

# Merge Together

1. Merge together all data
2. Apply resample

In [ ]:
data = pd.concat([ df , df_gt, df_ntl ] , axis=1).sort_index()
data.tail(5)

Resample the dataframe for all columns using the mean

In [ ]:
data = data.resample().mean()
data

Manage missing values for estimation

In [ ]:
data = data.dropna(how='any', axis=0)
data.head(10)

Apply growth-rate transformation

In [ ]:
data = data.pct_change().dropna()
data.head(10)

## Plot

Showing off more capabilities

In [ ]:
from scipy import stats

fig = plt.figure(figsize=(13, 8))
gs = fig.add_gridspec(2, 2, hspace=0.4, wspace=0.3)

# Time series — top row, full width
ax_ts = fig.add_subplot(gs[0, :])
ax_ts.plot(data['RGDP0000'].dropna(), color='#003865', linewidth=2,
           marker='o', markersize=4, label='GDP')
ax_ts.plot(data['gt_mean'].dropna(), color='#009CA7', linewidth=1.5,
           alpha=0.8, label='Google Trends')
ax_ts.plot(data['ntl_mean'].dropna(), color='#F5A623', linewidth=1.5,
           alpha=0.8, label='NTL')
ax_ts.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax_ts.set_title('Quarterly Growth Rates', fontsize=12, weight='bold')
ax_ts.set_ylabel('% change')
ax_ts.legend(frameon=False)
ax_ts.grid(axis='y', linestyle=':', alpha=0.5)

# Scatter: GDP vs Google Trends
ax_gt = fig.add_subplot(gs[1, 0])
common_gt = data[['RGDP0000', 'gt_mean']].dropna()
slope, intercept, r, *_ = stats.linregress(common_gt['gt_mean'], common_gt['RGDP0000'])
x_line = np.linspace(common_gt['gt_mean'].min(), common_gt['gt_mean'].max(), 100)
ax_gt.scatter(common_gt['gt_mean'], common_gt['RGDP0000'], color='#009CA7', alpha=0.6, s=40)
ax_gt.plot(x_line, intercept + slope * x_line, color='#009CA7', linewidth=1.5)
ax_gt.set_xlabel('Google Trends growth (%)')
ax_gt.set_ylabel('GDP growth (%)')
ax_gt.set_title(f'GDP vs Google Trends  (r = {r:.2f})', fontsize=11)
ax_gt.grid(linestyle=':', alpha=0.4)

# Scatter: GDP vs NTL
ax_ntl = fig.add_subplot(gs[1, 1])
common_ntl = data[['RGDP0000', 'ntl_mean']].dropna()
slope, intercept, r, *_ = stats.linregress(common_ntl['ntl_mean'], common_ntl['RGDP0000'])
x_line = np.linspace(common_ntl['ntl_mean'].min(), common_ntl['ntl_mean'].max(), 100)
ax_ntl.scatter(common_ntl['ntl_mean'], common_ntl['RGDP0000'], color='#F5A623', alpha=0.6, s=40)
ax_ntl.plot(x_line, intercept + slope * x_line, color='#F5A623', linewidth=1.5)
ax_ntl.set_xlabel('NTL growth (%)')
ax_ntl.set_ylabel('GDP growth (%)')
ax_ntl.set_title(f'GDP vs NTL  (r = {r:.2f})', fontsize=11)
ax_ntl.grid(linestyle=':', alpha=0.4)

fig.suptitle('Data Overview — Quarterly Growth Rates', fontsize=13, weight='bold')
plt.show()

## OLS Estimation

We want to estimate how well Google Trends and Nighttime Lights predict GDP growth:

$$\text{GDP growth}_t = \beta_0 + \beta_1 \cdot \text{Google Trends}_t + \beta_2 \cdot \text{NTL}_t + \varepsilon_t$$

There are two common Python libraries for OLS:

| Feature | sklearn | statsmodels |
| --- | --- | --- |
| Purpose | Machine learning / prediction | Statistical inference / econometrics |
| Output | Coefficients only | Coefs + SE, p-values, R2, CIs |
| Syntax | `LinearRegression().fit(X, y)` | `sm.OLS(y, X).fit()` |
| Use when | You want to predict | You want to interpret and test |

**sklearn:**
```python
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(X, y)
print(model.coef_)       # coefficients
print(model.intercept_)  # constant
```

**statsmodels** (used here):
```python
import statsmodels.api as sm
model = sm.OLS(y, sm.add_constant(X)).fit()
print(model.summary())
```

For nowcasting we use `statsmodels` because we care about *which indicators matter* and *by how much*, not just prediction.

In [ ]:
import statsmodels.api as sm

y = data['RGDP0000']
X = sm.add_constant(data[['gt_mean', 'ntl_mean']])

model = sm.OLS(y, X).fit()
print(model.summary())

## Coefficient Plot

In [ ]:
# Coefficient plot with capped confidence intervals
coefs = model.params.drop('const')
conf  = model.conf_int().drop('const')

colors = {'gt_mean': '#009CA7', 'ntl_mean': '#F5A623'}
labels = {'gt_mean': 'Google Trends', 'ntl_mean': 'NTL'}

fig, ax = plt.subplots(figsize=(8, 3))

for i, var in enumerate(coefs.index):
    ax.errorbar(
        coefs[var], i,
        xerr=[[coefs[var] - conf.loc[var, 0]], [conf.loc[var, 1] - coefs[var]]],
        fmt='o', color=colors[var], capsize=7, capthick=2.5,
        markersize=8, linewidth=2.5, zorder=3
    )
    sig = '***' if model.pvalues[var] < 0.01 else ('**' if model.pvalues[var] < 0.05 else ('*' if model.pvalues[var] < 0.1 else ''))
    ax.text(conf.loc[var, 1] + 0.002, i, f"{coefs[var]:.3f}{sig}",
            va='center', fontsize=10, color=colors[var])

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(range(len(coefs)))
ax.set_yticklabels([labels[v] for v in coefs.index], fontsize=11)
ax.set_xlabel('Coefficient (effect on GDP quarterly growth %)')
ax.set_title(f'OLS Coefficients — 95% CI   (R² = {model.rsquared:.2f})',
             fontsize=12, weight='bold')
ax.grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## Actual vs Fitted

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(y.index, y, color='#003865', linewidth=2,
        marker='o', markersize=4, label='Actual GDP growth')
ax.plot(model.fittedvalues.index, model.fittedvalues,
        color='#003865', linewidth=1.5, linestyle='--',
        alpha=0.7, label='Fitted')
ax.fill_between(y.index, y, model.fittedvalues,
                alpha=0.12, color='#003865', label='Residual')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_title('GDP Growth — Actual vs Fitted', fontsize=12, weight='bold')
ax.set_ylabel('% change')
ax.legend(frameon=False)
ax.grid(axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## Standardized Coefficients

Running the same regression on **standardized variables** (each variable minus its mean, divided by its standard deviation).

A standardized coefficient tells you: *how many standard deviations does GDP growth change when the predictor increases by one standard deviation?*

This makes the two coefficients directly comparable, even though Google Trends and NTL are measured in completely different units.

In [ ]:
data_std = (data - data.mean()) / data.std()

y_std = data_std['RGDP0000']
X_std = sm.add_constant(data_std[['gt_mean', 'ntl_mean']])

model_std = sm.OLS(y_std, X_std).fit()
print(model_std.summary())